[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Inheritance &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The cell below rebuilds the two classes the tasks start from. Run it first.


In [1]:
class Station:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r})"

    def mean(self):
        if not self.readings:
            return None
        return round(sum(self.readings) / len(self.readings), 2)

    def report(self):
        return f"{self.name}: mean {self.mean()}"

    @classmethod
    def from_csv(cls, line):
        name, *values = line.split(",")
        return cls(name, [float(v) for v in values])


class AutomaticStation(Station):
    def report(self):
        return super().report() + " (automatic, unattended)"


print("ready")


ready


**1.** A subclass that extends `__init__`.


In [2]:
class MountainStation(Station):
    def __init__(self, name, readings, altitude):
        super().__init__(name, readings)
        self.altitude = altitude


mountain = MountainStation("Galdhopiggen", [-9.4, -11.2], altitude=2469)

print(vars(mountain))


{'name': 'Galdhopiggen', 'readings': [-9.4, -11.2], 'altitude': 2469}


`super().__init__(name, readings)` stored the first two attributes with `Station`'s own code, and the
last line stored the one that is new. The object ends up holding all three, as though one `__init__`
had written them.


**2.** An override that reuses the original.


In [3]:
class MountainStation(Station):
    def __init__(self, name, readings, altitude):
        super().__init__(name, readings)
        self.altitude = altitude

    def report(self):
        return f"{super().report()}, at {self.altitude} m"


mountain = MountainStation("Galdhopiggen", [-9.4, -11.2], altitude=2469)

print(mountain.report())


Galdhopiggen: mean -10.3, at 2469 m


The override adds one thing and hands the rest to `Station.report`. If `Station` later changes how it
formats a mean, the mountain report changes with it.


**3.** `isinstance` against `type(...) is`.


In [4]:
print("isinstance(mountain, Station):", isinstance(mountain, Station))
print("type(mountain) is Station:    ", type(mountain) is Station)

# isinstance asks whether mountain is a Station of any kind, and a MountainStation
# is one, because MountainStation is built on Station. type(...) is asks whether
# its class is exactly Station, and its class is MountainStation.


isinstance(mountain, Station): True
type(mountain) is Station:     False


Code that checks with `isinstance` accepts every future subclass without being edited. Code that
checks with `type(...) is` has to be changed each time a new subclass appears.


**4.** The search order, and where `mean` comes from.


In [5]:
print([c.__name__ for c in MountainStation.__mro__])

for owner in MountainStation.__mro__:
    if "mean" in vars(owner):
        print("mean is supplied by", owner.__name__)
        break


['MountainStation', 'Station', 'object']
mean is supplied by Station


`MountainStation` is searched first and has no `mean`, so the search moves to `Station`, which does.
`object` is last in every list, and is never reached for `mean`.


**5.** An inherited constructor, building the subclass.


In [6]:
svalbard = AutomaticStation.from_csv("Svalbard,-12.5,-14.0")

print(type(svalbard).__name__)
print(svalbard.report())

# from_csv is defined on Station, but it was called on AutomaticStation, so cls
# was AutomaticStation, and cls(...) built one. Had from_csv written
# Station(...) instead, it would have built a plain station whatever it was
# called on, and the automatic report would be lost.


AutomaticStation
Svalbard: mean -13.25 (automatic, unattended)


This is the promise the **Class and Static Methods** notebook made, and it only holds because
`from_csv` writes `cls(...)`.


**6.** One loop, three kinds of station.


In [7]:
network = [
    Station("Tromso", [-4.1, -2.6]),
    MountainStation("Galdhopiggen", [-9.4, -11.2], altitude=2469),
    AutomaticStation.from_csv("Svalbard,-12.5,-14.0"),
]

for station in network:
    print(f"{station!r:<32} {station.report()}")


Station('Tromso')                Tromso: mean -3.35
MountainStation('Galdhopiggen')  Galdhopiggen: mean -10.3, at 2469 m
AutomaticStation('Svalbard')     Svalbard: mean -13.25 (automatic, unattended)


The loop calls `report()` and nothing else, and each station answers with its own class's version.
Adding a fourth kind of station later needs a new subclass and no change to this loop.


---

&#8592; **Back to:** [Inheritance](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/09-inheritance.ipynb)  &nbsp;&middot;&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)
